# Retrieval results — narrative figures (formula, PubChem-all, window ablations, unions)

Replaces the MW->RI funnel notebook's dead end (`fig_retrieval_results_mw_funnel.ipynb`
showed MW->RI funneling doesn't beat RI-only — a null result, not shown again here)
with four plot families that tell the actual retrieval story:

1. **Formula retrieval** and **PubChem-all retrieval** top-k-vs-k curves —
   the two clean, standard baseline comparisons (ICICLE vs. NEIMS/RASSP/MassFormer).
2. **PubChem-all top-k-vs-k, ICICLE only, MW/heavy-atom window traces overlaid** —
   same base retrieval, each auxiliary filter drawn as a translucent shade of
   ICICLE's own color (not a new color) since it's the same model under a
   different candidate restriction, not a different model.
3. **Top-k-vs-k for RI-restricted queries, with MW-union-RI and HA-union-RI
   traces** — does combining RI with an independent structural filter help,
   for the subset of queries that actually have an RI value?
4. **Top-1 accuracy vs. candidate-set size** — RI-only ladder plus MW-union
   and heavy-atom-union ladders at the same candidate-set sizes (this is
   what the old funnel notebook showed, minus the funnel strategy that
   didn't pan out).

All figures: square format, no titles (see summary tables + this notebook's
own section headers for context), one fixed color per model, translucent
shades + alternate markers for within-model filter variants. `mode="autofail"`
throughout per project convention (no ground-truth injection).

Heavy-atom-union results are still running for some models as of this
notebook's creation — cells below skip missing files gracefully and print
what's missing, so re-running later fills in new panels without editing code.

In [ ]:
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from icicle.utils.visualization.eval_plots import (
    load_seed_csvs,
    model_color,
    plot_ladder_multi_strategy,
    plot_topk_curves,
)
from icicle.utils.visualization.style import make_fig, save_fig, set_style

set_style("manuscript")

RESULTS = Path("/home/magled/icicle-dev/results")
OUTPUT_DIR = Path("figures/retrieval_narrative")
MODE = "autofail"
K_VALUES = [1, 2, 3, 4, 5, 10, 15, 20, 30, 40, 50]
SUMMARY_K_VALUES = [1, 5, 10, 20, 50]
RI_TYPE = "StdNP"  # primary RI type used throughout (largest query subset)

## 1a. Formula retrieval — top-k accuracy vs. k

ICICLE vs. NEIMS/RASSP/MassFormer, scaffold split, 3 seeds, mean +/- 95% CI.
Same data source as `fig_retrieval_results_formula_match.ipynb`'s "scaffold"
comparison.

In [ ]:
FORMULA_MODEL_SEED_DIRS = {
    "ICICLE": [
        RESULTS / "eval" / f"final_entropy_scaffold_s{i}_retr"
        for i in (1, 2, 3)
    ],
    "NEIMS": [RESULTS / "eval" / f"neims_scaffold_s{i}" for i in (1, 2, 3)],
    "RASSP": [RESULTS / "eval" / f"rassp_scaffold_s{i}" for i in (1, 2, 3)],
    "MassFormer": [
        RESULTS / "eval" / f"massformer_scaffold_s{i}" for i in (1, 2, 3)
    ],
}

formula_model_dfs = {}
for label, dirs in FORMULA_MODEL_SEED_DIRS.items():
    dfs = load_seed_csvs(
        [d / "retrieval_with_formula_results.csv" for d in dirs]
    )
    if dfs:
        formula_model_dfs[label] = dfs
    else:
        print(f"{label}: no formula-retrieval CSVs found, skipping")

In [ ]:
fig = plot_topk_curves(
    formula_model_dfs, rank_col="rank_cosine_similarity", k_values=K_VALUES
)
save_fig(fig, "retrieval_formula_topk_vs_k", OUTPUT_DIR)
plt.show()
plt.close(fig)

## 1b. PubChem-all retrieval — top-k accuracy vs. k

Candidates = entire PubChem (~93M molecules), no formula/RI restriction.
Single run per model (no seed replication), `mode="autofail"`.

In [ ]:
PUBCHEM_ALL_PATHS = {
    "ICICLE": RESULTS
    / "pubchem_retrieval_eval_icicle_rerun_260710"
    / "retrieval_per_query_StdNP_all.tsv",
    "NEIMS": RESULTS
    / "pubchem_retrieval_eval_neims"
    / "retrieval_per_query_StdNP_all.tsv",
    "MassFormer": RESULTS
    / "pubchem_retrieval_eval_massformer"
    / "retrieval_per_query_StdNP_all.tsv",
}

pubchem_all_dfs = {}
for label, path in PUBCHEM_ALL_PATHS.items():
    if path.exists():
        pubchem_all_dfs[label] = pd.read_csv(path, sep="\t")
    else:
        print(f"{label}: {path} not found, skipping")


def topk_accuracy_curve(df, rank_col, k_values):
    ranks = df[rank_col].dropna()
    return [100.0 * (ranks <= k).mean() for k in k_values]


def plot_pubchem_topk_curves(model_dfs, rank_col, k_values):
    fig, ax = make_fig("square")
    for i, (label, df) in enumerate(model_dfs.items()):
        if rank_col not in df.columns:
            continue
        accs = topk_accuracy_curve(df, rank_col, k_values)
        ax.plot(
            k_values,
            accs,
            marker="o",
            label=label,
            color=model_color(label, fallback_index=i),
        )
    ax.set_xlabel("k")
    ax.set_ylabel("Top-k accuracy (%)")
    ax.set_ylim(0, 100)
    ax.legend()
    return fig

In [ ]:
rank_col = f"rank_{MODE}_cosine"
dfs_with_mode = {
    label: df
    for label, df in pubchem_all_dfs.items()
    if rank_col in df.columns
}
fig = plot_pubchem_topk_curves(dfs_with_mode, rank_col, K_VALUES)
save_fig(fig, f"retrieval_pubchem_all_topk_vs_k_{MODE}", OUTPUT_DIR)
plt.show()
plt.close(fig)

## 2. PubChem-all top-k-vs-k, ICICLE only — MW and heavy-atom window traces overlaid

Same base retrieval as 1b (candidates = full PubChem, no RI restriction),
but for ICICLE only: RI-only/no-filter line, plus each MW window
(`retrieval_mw*_global_results.json`) and each heavy-atom window
(`retrieval_heavy_atom*_global_results.json`) as a translucent shade of
ICICLE's color with a distinct marker. These are *global* (no candidate-set-
size ladder, no RI restriction) filters, so they plot as flat top-k-vs-k
curves rather than points on a ladder.

In [ ]:
ICICLE_DIR = RESULTS / "pubchem_retrieval_eval_icicle_rerun_260710"

MW_GLOBAL_TAGS = {
    "MW +-80Da": "80",
    "MW [-10,+80]Da": "10_80",
    "MW +-10Da": "10_10",
    "MW +-5Da": "5",
}
HA_GLOBAL_WINDOWS = [1, 2, 3, 6, 8]

global_settings = {"No filter (full PubChem)": pubchem_all_dfs.get("ICICLE")}

mw_global_metrics = {}
for label, tag in MW_GLOBAL_TAGS.items():
    path = ICICLE_DIR / f"retrieval_mw{tag}_global_results.json"
    if path.exists():
        with open(path) as f:
            mw_global_metrics[label] = json.load(f)["all"][MODE]["cosine"]
    else:
        print(f"missing {path}")

ha_global_metrics = {}
for w in HA_GLOBAL_WINDOWS:
    path = ICICLE_DIR / f"retrieval_heavy_atom{w}_global_results.json"
    if path.exists():
        with open(path) as f:
            ha_global_metrics[f"HA +-{w}"] = json.load(f)["all"][MODE][
                "cosine"
            ]
    else:
        print(f"missing {path}")

In [ ]:
def plot_topk_vs_k_shaded(
    base_label, base_df, base_rank_col, k_values, shaded_metrics, base_color
):
    """Top-k-vs-k for one model's base retrieval, plus flat-metric-dict shades."""
    fig, ax = make_fig("square")
    markers = ["s", "^", "D", "v", "P", "X", "*"]
    alphas = [0.55, 0.45, 0.4, 0.35, 0.3, 0.3, 0.3]

    if base_df is not None:
        accs = topk_accuracy_curve(base_df, base_rank_col, k_values)
        ax.plot(k_values, accs, marker="o", label=base_label, color=base_color)

    for i, (label, m) in enumerate(shaded_metrics.items()):
        y = [
            100.0 * m[f"top_{k}_accuracy"]
            for k in k_values
            if f"top_{k}_accuracy" in m
        ]
        x = [k for k in k_values if f"top_{k}_accuracy" in m]
        if not y:
            continue
        ax.plot(
            x,
            y,
            marker=markers[i % len(markers)],
            label=label,
            color=base_color,
            alpha=alphas[i % len(alphas)],
        )

    ax.set_xlabel("k")
    ax.set_ylabel("Top-k accuracy (%)")
    ax.set_ylim(0, 100)
    ax.legend(fontsize="small", loc="lower right")
    return fig

In [ ]:
icicle_color = model_color("ICICLE")
rank_col = f"rank_{MODE}_cosine"

fig = plot_topk_vs_k_shaded(
    "ICICLE (no filter)",
    pubchem_all_dfs.get("ICICLE"),
    rank_col,
    SUMMARY_K_VALUES,
    {**mw_global_metrics, **ha_global_metrics},
    icicle_color,
)
save_fig(fig, f"retrieval_icicle_pubchem_all_window_traces_{MODE}", OUTPUT_DIR)
plt.show()
plt.close(fig)

## 3. RI-restricted queries — top-k-vs-k with MW-union-RI and HA-union-RI traces

Restricted to queries that have an RI value (`{RI_TYPE}` column type).
Base line = RI-only ladder at its largest available level (all StdNP
queries, RI-only ranking). Overlay = MW-union-RI and heavy-atom-union-RI
(`retrieval_union_mw*_results.json` / heavy-atom equivalent, once that job
finishes) at the matching level, same translucent-shade convention.
ICICLE-only (the union job is model-specific and still running for some
models); re-run once NEIMS/MassFormer union files land to extend.

In [ ]:
UNION_LEVEL = "all"  # largest available level for the union ladder

ri_only_path = ICICLE_DIR / "retrieval_ablation_all_ri_types.json"
ri_only_ladder = {}
if ri_only_path.exists():
    with open(ri_only_path) as f:
        ri_only_ladder = json.load(f)[RI_TYPE]

mw_union_metrics = {}
for label, tag in MW_GLOBAL_TAGS.items():
    path = ICICLE_DIR / f"retrieval_union_mw{tag}_results.json"
    if path.exists():
        with open(path) as f:
            d = json.load(f)
        levels = d.get(RI_TYPE, {})
        level = (
            UNION_LEVEL
            if UNION_LEVEL in levels
            else (
                max(
                    (lvl for lvl in levels if lvl != "all"),
                    key=int,
                    default=None,
                )
            )
        )
        if level is not None:
            mw_union_metrics[f"MW union ({label})"] = levels[level][MODE][
                "cosine"
            ]
    else:
        print(f"missing {path}")

ha_union_metrics = {}
for w in HA_GLOBAL_WINDOWS:
    path = ICICLE_DIR / f"retrieval_union_heavy_atom{w}_results.json"
    if path.exists():
        with open(path) as f:
            d = json.load(f)
        levels = d.get(RI_TYPE, {})
        level = (
            UNION_LEVEL
            if UNION_LEVEL in levels
            else (
                max(
                    (lvl for lvl in levels if lvl != "all"),
                    key=int,
                    default=None,
                )
            )
        )
        if level is not None:
            ha_union_metrics[f"HA union (+-{w})"] = levels[level][MODE][
                "cosine"
            ]
    else:
        print(f"missing {path} (heavy-atom union still running)")

In [ ]:
ri_only_base = None
if RI_TYPE in ri_only_ladder:
    levels = ri_only_ladder
    level = (
        UNION_LEVEL
        if UNION_LEVEL in levels
        else max(
            (lvl for lvl in levels if lvl != "all"), key=int, default=None
        )
    )
    if level is not None:
        ri_only_base = levels[level][MODE]["cosine"]

fig, ax = make_fig("square")
markers = ["s", "^", "D", "v", "P", "X", "*"]
alphas = [0.55, 0.45, 0.4, 0.35, 0.3, 0.3, 0.3]

if ri_only_base is not None:
    y = [100.0 * ri_only_base[f"top_{k}_accuracy"] for k in SUMMARY_K_VALUES]
    ax.plot(
        SUMMARY_K_VALUES,
        y,
        marker="o",
        label=f"RI-only ({RI_TYPE})",
        color=icicle_color,
    )

for i, (label, m) in enumerate(
    {**mw_union_metrics, **ha_union_metrics}.items()
):
    y = [
        100.0 * m[f"top_{k}_accuracy"]
        for k in SUMMARY_K_VALUES
        if f"top_{k}_accuracy" in m
    ]
    x = [k for k in SUMMARY_K_VALUES if f"top_{k}_accuracy" in m]
    if not y:
        continue
    ax.plot(
        x,
        y,
        marker=markers[i % len(markers)],
        label=label,
        color=icicle_color,
        alpha=alphas[i % len(alphas)],
    )

ax.set_xlabel("k")
ax.set_ylabel("Top-k accuracy (%)")
ax.set_ylim(0, 100)
ax.legend(fontsize="small", loc="lower right")
save_fig(fig, f"retrieval_icicle_ri_union_traces_{RI_TYPE}_{MODE}", OUTPUT_DIR)
plt.show()
plt.close(fig)

## 4. Top-1 accuracy vs. candidate-set size — RI-only ladder + MW-union + HA-union

The direct replacement for the old MW-funnel notebook's headline plot,
minus the funnel strategy (shown not to help). One figure per model: the
RI-only ladder plus MW-union and heavy-atom-union ladders at the same
candidate-set sizes, all in that model's color at increasing translucency.

In [ ]:
MODEL_DIRS = {
    "ICICLE": RESULTS / "pubchem_retrieval_eval_icicle_rerun_260710",
    "NEIMS": RESULTS / "pubchem_retrieval_eval_neims",
    "MassFormer": RESULTS / "pubchem_retrieval_eval_massformer",
}
K_FOR_LADDER = 1

for model_label, model_dir in MODEL_DIRS.items():
    strategies = {}

    ladder_path = model_dir / "retrieval_ablation_all_ri_types.json"
    if ladder_path.exists():
        with open(ladder_path) as f:
            ladder = json.load(f)
        if RI_TYPE in ladder:
            strategies["RI-only"] = {
                lvl: v for lvl, v in ladder[RI_TYPE].items() if lvl != "all"
            }

    mw_tag = MW_GLOBAL_TAGS["MW [-10,+80]Da"]
    mw_union_path = model_dir / f"retrieval_union_mw{mw_tag}_results.json"
    if mw_union_path.exists():
        with open(mw_union_path) as f:
            d = json.load(f)
        if RI_TYPE in d:
            strategies["MW[-10,+80]Da union RI"] = d[RI_TYPE]
    else:
        print(f"{model_label}: missing {mw_union_path}")

    for w in [3]:
        ha_union_path = (
            model_dir / f"retrieval_union_heavy_atom{w}_results.json"
        )
        if ha_union_path.exists():
            with open(ha_union_path) as f:
                d = json.load(f)
            if RI_TYPE in d:
                strategies[f"HA(+-{w}) union RI"] = d[RI_TYPE]
        else:
            print(f"{model_label}: missing {ha_union_path} (still running)")

    if not strategies:
        print(f"{model_label}: no ladder data at all, skipping")
        continue

    fig = plot_ladder_multi_strategy(
        strategies,
        metric="cosine",
        mode=MODE,
        k=K_FOR_LADDER,
        base_color=model_color(model_label),
    )
    save_fig(
        fig,
        f"retrieval_ladder_unions_{model_label}_{RI_TYPE}_top{K_FOR_LADDER}_{MODE}",
        OUTPUT_DIR,
    )
    plt.show()
    plt.close(fig)

## Summary table

All settings shown above (formula, PubChem-all, MW/HA global windows,
MW/HA-union-RI, RI-only ladder), one row per (model, setting, level, mode),
top-k accuracies + MRR + median rank where available.

In [ ]:
rows = []

for label, dfs in formula_model_dfs.items():
    df = dfs[0]
    row = {
        "model": label,
        "setting": "Formula retrieval",
        "level": "all",
        "mode": "n/a",
    }
    ranks = df["rank_cosine_similarity"]
    for k in SUMMARY_K_VALUES:
        row[f"top-{k}"] = f"{100 * (ranks <= k).mean():.1f}"
    rows.append(row)

for label, df in pubchem_all_dfs.items():
    rank_col = f"rank_{MODE}_cosine"
    if rank_col not in df.columns:
        continue
    ranks = df[rank_col].dropna()
    row = {
        "model": label,
        "setting": "PubChem-all",
        "level": "all",
        "mode": MODE,
    }
    for k in SUMMARY_K_VALUES:
        row[f"top-{k}"] = f"{100 * (ranks <= k).mean():.1f}"
    rows.append(row)

for label, m in {**mw_global_metrics, **ha_global_metrics}.items():
    row = {
        "model": "ICICLE",
        "setting": label,
        "level": "all (global)",
        "mode": MODE,
    }
    for k in SUMMARY_K_VALUES:
        if f"top_{k}_accuracy" in m:
            row[f"top-{k}"] = f"{100 * m[f'top_{k}_accuracy']:.1f}"
    row["mrr"] = f"{m['mrr']:.4f}"
    row["median_rank"] = m["median_rank"]
    rows.append(row)

for label, m in {**mw_union_metrics, **ha_union_metrics}.items():
    row = {
        "model": "ICICLE",
        "setting": label,
        "level": UNION_LEVEL,
        "mode": MODE,
    }
    for k in SUMMARY_K_VALUES:
        if f"top_{k}_accuracy" in m:
            row[f"top-{k}"] = f"{100 * m[f'top_{k}_accuracy']:.1f}"
    row["mrr"] = f"{m['mrr']:.4f}"
    row["median_rank"] = m["median_rank"]
    rows.append(row)

df_summary = pd.DataFrame(rows)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(OUTPUT_DIR / "retrieval_narrative_summary.csv", index=False)
df_summary